In [ ]:
# ===== 패키지 설치 (최초 1회만 실행) =====
!pip install python-dotenv
!pip install -U langchain langchain-openai langchain-teddynote

# ===== 필요한 모듈 import =====
import os
from dotenv import load_dotenv
from langchain_teddynote import logging
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, load_prompt
from datetime import datetime

# ===== 환경변수(.env) 로드 =====
load_dotenv()  # override=False가 기본값이므로 시스템 환경변수가 우선순위를 가짐

# (선택) 정상 로드 확인
print("OpenAI 키:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("LANGSMITH 키:", os.getenv("LANGSMITH_API_KEY")[:8] + "...")
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

# ===== LangSmith 추적 시작 =====
logging.langsmith("CH02-Prompt")  # 프로젝트명 입력

# ===== LLM 객체 생성 =====
llm = ChatOpenAI()

### 프롬프트 템플릿 만들기
- 당신은 질문-답변(Question-Answer) Task를 수행하는 AI 어시스턴트입니다.
- 검색된 문맥(context)을 사용하여 질문(question)에 답하세요.
- 만약, 문맥(context)으로부터 답을 찾을 수 없다면 '모른다'고 말하세요.

- 한국어로 대답하세요.
- Question: {이곳에 사용자가 입력한 질문이 삽입됩니다}
- Context: {이곳에 검색된 정보가 삽입됩니다}

In [ ]:
# template 정의. {country}는 변수로, 이후에 값이 들어갈 자리를 의미
template = "{country}의 수도는 어디인가요?"

# from_template 메서드를 이용하여 PromptTemplate 객체 생성
prompt = PromptTemplate.from_template(template)
prompt

In [4]:
# prompt 생성. format 메서드를 이용하여 변수에 값을 넣어줌
prompt = prompt.format(country="대한민국")
prompt

'대한민국의 수도는 어디인가요?'

In [5]:
template = "{country}의 수도는 어디인가요?" # template 정의

# from_template 메서드를 이용하여 PromptTemplate 객체 생성
prompt = PromptTemplate.from_template(template)

chain = prompt | llm # chain 생성

# country 변수에 입력된 값이 자동으로 치환되어 수행됨
chain.invoke("대한민국").content

'대한민국의 수도는 서울입니다.'

In [6]:
template = "{country}의 수도는 어디인가요?" # template 정의

# PromptTemplate 객체를 활용하여 prompt_template 생성
prompt = PromptTemplate(
    template=template,
    input_variables=["country"],
)

prompt

PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디인가요?')

In [7]:
prompt.format(country="대한민국")

'대한민국의 수도는 어디인가요?'

In [10]:
template = "{country1}과 {country2}의 수도는 각각 어디인가요?" # template 정의

# PromptTemplate 객체를 활용하여 prompt_template 생성
prompt = PromptTemplate(
    template=template,
    input_variables=["country1"],
    partial_variables={
        "country2": "미국" # dictionary 형태로 partial_variables를 전달
    },
)

prompt

PromptTemplate(input_variables=['country1'], input_types={}, partial_variables={'country2': '미국'}, template='{country1}과 {country2}의 수도는 각각 어디인가요?')

In [11]:
prompt.format(country1="대한민국")

'대한민국과 미국의 수도는 각각 어디인가요?'

In [12]:
prompt_partial = prompt.partial(country2="캐나다")
prompt_partial

PromptTemplate(input_variables=['country1'], input_types={}, partial_variables={'country2': '캐나다'}, template='{country1}과 {country2}의 수도는 각각 어디인가요?')

In [13]:
prompt_partial.format(country1="대한민국")

'대한민국과 캐나다의 수도는 각각 어디인가요?'

In [14]:
chain = prompt_partial | llm

chain.invoke("대한민국").content

'대한민국의 수도는 서울이며, 캐나다의 수도는 오타와입니다.'

In [15]:
chain.invoke({"country1": "대한민국", "country2": "호주"}).content

'대한민국의 수도는 서울이고, 호주의 수도는 캔버라입니다.'